# Cluster ↔ Video Performance Correlation

This notebook measures how strongly within-channel semantic clusters (from category quantization output) are associated with video performance. It computes a documented metric set per channel, and produces an ordered ranking from **most predictive** to **least predictive** using a primary metric.


## 1) Setup and inputs

This cell imports analysis dependencies and defines paths for the category quantization JSON export and optional CSV outputs.


In [5]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = Path('/content/drive/MyDrive/Graphiko/exports/video_embeddings_clustered/latest/business_cluster_video_embeddings_clustered_2d.json')
OUTPUT_DIR = Path('/content/drive/MyDrive/Graphiko/analysis/cluster_performance')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PRIMARY_ORDERING_METRIC = 'adj_r2'  # Used to rank channels from most to less predictive.
MIN_VIDEOS_PER_CHANNEL = 10
MIN_CLUSTERS_PER_CHANNEL = 2


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2) Load and validate exported cluster JSON

This cell loads the category-quantization JSON and validates required columns so the downstream metrics are reproducible and explicit.


In [6]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Expected JSON export at: {DATA_PATH.resolve()}")

df = pd.read_json(DATA_PATH)
required_cols = {"channel_name", "cluster_id", "cluster_name", "view_count", "video_title", "video_url"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

# Ensure numeric performance target and remove missing rows that block modeling.
df["view_count"] = pd.to_numeric(df["view_count"], errors="coerce")
df = df.dropna(subset=["channel_name", "cluster_id", "view_count"]).copy()
df["cluster_id"] = df["cluster_id"].astype(str)

# Log-transform to stabilize heavy-tailed view distributions.
df["log_views"] = np.log1p(df["view_count"])

print(f"Rows loaded: {len(df):,}")
print(f"Channels: {df['channel_name'].nunique():,}")
print(df.head(3))


Rows loaded: 1,344
Channels: 27
                channel_name  \
0  20VC with Harry Stebbings   
1  20VC with Harry Stebbings   
2  20VC with Harry Stebbings   

                                         video_title  \
0  Jake Paul: Traditional VC is Toast & Attention...   
1  SpaceX's Financials Leaked: Is it Worth $2TN |...   
2  The Early Days of Anthropic & How 21 of 22 VCs...   

                                     video_url  view_count  viewCount  \
0  https://www.youtube.com/watch?v=rWn3KgO9Dvk        6910       6910   
1  https://www.youtube.com/watch?v=UOY8hsBqjJo       21482      21482   
2  https://www.youtube.com/watch?v=a1ymdW-h33E       26531      26531   

  cluster_id                            cluster_name     video_id  \
0          3  20VC with Harry Stebbings | Category 4  rWn3KgO9Dvk   
1          1  20VC with Harry Stebbings | Category 2  UOY8hsBqjJo   
2          2  20VC with Harry Stebbings | Category 3  a1ymdW-h33E   

                 channel_id                 

## 3) Metric definitions

This cell defines one function that computes a full channel-level metric bundle:

- **ANOVA p-value**: whether mean log-views differ across clusters.
- **R² / Adjusted R²** from `log_views ~ C(cluster_id)`.
- **Eta-squared (η²)**: ANOVA effect size = between-cluster sum of squares / total sum of squares.
- **Kruskal-Wallis p-value**: non-parametric alternative to ANOVA.
- **Epsilon-squared (ε²)** for Kruskal: non-parametric effect size.
- **Cluster median spread ratio**: max cluster median / min cluster median (on raw views).

The notebook uses **Adjusted R² (`adj_r2`)** as the primary ordering metric.


In [7]:
def channel_metrics(channel_df: pd.DataFrame) -> dict:
    channel_name = channel_df["channel_name"].iloc[0]
    n_videos = len(channel_df)
    n_clusters = channel_df["cluster_id"].nunique()

    # Guardrails for statistical validity.
    if n_videos < MIN_VIDEOS_PER_CHANNEL or n_clusters < MIN_CLUSTERS_PER_CHANNEL:
        return {
            "channel_name": channel_name,
            "n_videos": n_videos,
            "n_clusters": n_clusters,
            "anova_pvalue": np.nan,
            "r2": np.nan,
            "adj_r2": np.nan,
            "eta_squared": np.nan,
            "kruskal_pvalue": np.nan,
            "epsilon_squared": np.nan,
            "median_spread_ratio": np.nan,
            "eligible": False,
            "note": "Insufficient videos or clusters"
        }

    model = smf.ols("log_views ~ C(cluster_id)", data=channel_df).fit()
    anova_tbl = sm.stats.anova_lm(model, typ=2)

    ss_between = float(anova_tbl.loc["C(cluster_id)", "sum_sq"])
    ss_total = ss_between + float(anova_tbl.loc["Residual", "sum_sq"])
    eta_squared = (ss_between / ss_total) if ss_total > 0 else np.nan
    anova_p = float(anova_tbl.loc["C(cluster_id)", "PR(>F)"])

    groups_log = [g["log_views"].values for _, g in channel_df.groupby("cluster_id")]
    kw_stat, kw_p = stats.kruskal(*groups_log)

    k = n_clusters
    n = n_videos
    epsilon_sq = (kw_stat - k + 1) / (n - k) if (n - k) > 0 else np.nan

    cluster_medians = channel_df.groupby("cluster_id")["view_count"].median()
    min_med = cluster_medians.min()
    max_med = cluster_medians.max()
    spread_ratio = (max_med / min_med) if min_med > 0 else np.nan

    return {
        "channel_name": channel_name,
        "n_videos": n_videos,
        "n_clusters": n_clusters,
        "anova_pvalue": anova_p,
        "r2": float(model.rsquared),
        "adj_r2": float(model.rsquared_adj),
        "eta_squared": eta_squared,
        "kruskal_pvalue": float(kw_p),
        "epsilon_squared": float(epsilon_sq),
        "median_spread_ratio": float(spread_ratio) if pd.notna(spread_ratio) else np.nan,
        "eligible": True,
        "note": "ok"
    }


## 4) Compute metrics for each channel

This cell runs the metric function across all channels and creates a complete channel-level metrics table.


In [8]:
metrics_df = pd.DataFrame([
    channel_metrics(g)
    for _, g in df.groupby("channel_name", sort=True)
])

metrics_df = metrics_df.sort_values(["eligible", PRIMARY_ORDERING_METRIC], ascending=[False, False])
metrics_df.reset_index(drop=True, inplace=True)

metrics_df.head(10)


,channel_name,n_videos,n_clusters,anova_pvalue,r2,adj_r2,eta_squared,kruskal_pvalue,epsilon_squared,median_spread_ratio,eligible,note
0,Real Vision Presents,50,9,3.315939e-08,0.685282,0.623873,0.685282,0.000061,0.610120,148.020147,True,ok
1,Network State Podcast,50,4,8.863858e-09,0.580662,0.553314,0.580662,0.000004,0.538813,15.885296,True,ok
2,Bg2 Pod,44,4,1.070268e-06,0.529695,0.494422,0.529695,0.020887,0.168561,12.974364,True,ok
3,Joe Lonsdale,50,4,1.366379e-04,0.356025,0.314027,0.356025,0.000460,0.323998,8.647866,True,ok
4,Tony Robbins,50,4,2.466463e-04,0.338632,0.295499,0.338632,0.000234,0.354929,3.451219,True,ok
5,Alex Hormozi,50,9,8.234211e-03,0.375282,0.253386,0.375282,0.030042,0.219668,8.677347,True,ok
6,Patrick Boyle,50,7,3.598111e-02,0.259289,0.155934,0.259289,0.042082,0.164203,2.280957,True,ok
7,Valuetainment,50,5,2.054038e-02,0.223117,0.154061,0.223117,0.015296,0.184304,6.851796,True,ok
8,My First Million,50,4,2.051923e-02,0.189749,0.136907,0.189749,0.042263,0.112804,3.526558,True,ok
9,Greg Isenberg,50,9,8.326713e-02,0.272301,0.130311,0.272301,0.135153,0.106760,4.018366,True,ok


## 5) Ordered list of channels (most predictive → less predictive)

This cell displays the ranked channel list using **Adjusted R²** as the primary metric, while keeping the full metric set visible.


In [9]:
ranked = metrics_df[metrics_df["eligible"]].copy()
ranked = ranked.sort_values(PRIMARY_ORDERING_METRIC, ascending=False).reset_index(drop=True)
ranked.insert(0, "rank", np.arange(1, len(ranked) + 1))

display_cols = [
    "rank", "channel_name", "n_videos", "n_clusters",
    "adj_r2", "r2", "eta_squared", "anova_pvalue",
    "epsilon_squared", "kruskal_pvalue", "median_spread_ratio"
]

ranked[display_cols]


,rank,channel_name,n_videos,n_clusters,adj_r2,r2,eta_squared,anova_pvalue,epsilon_squared,kruskal_pvalue,median_spread_ratio
0,1,Real Vision Presents,50,9,0.623873,0.685282,0.685282,3.315939e-08,0.610120,0.000061,148.020147
1,2,Network State Podcast,50,4,0.553314,0.580662,0.580662,8.863858e-09,0.538813,0.000004,15.885296
2,3,Bg2 Pod,44,4,0.494422,0.529695,0.529695,1.070268e-06,0.168561,0.020887,12.974364
3,4,Joe Lonsdale,50,4,0.314027,0.356025,0.356025,1.366379e-04,0.323998,0.000460,8.647866
4,5,Tony Robbins,50,4,0.295499,0.338632,0.338632,2.466463e-04,0.354929,0.000234,3.451219
5,6,Alex Hormozi,50,9,0.253386,0.375282,0.375282,8.234211e-03,0.219668,0.030042,8.677347
6,7,Patrick Boyle,50,7,0.155934,0.259289,0.259289,3.598111e-02,0.164203,0.042082,2.280957
7,8,Valuetainment,50,5,0.154061,0.223117,0.223117,2.054038e-02,0.184304,0.015296,6.851796
8,9,My First Million,50,4,0.136907,0.189749,0.189749,2.051923e-02,0.112804,0.042263,3.526558
9,10,Greg Isenberg,50,9,0.130311,0.272301,0.272301,8.326713e-02,0.106760,0.135153,4.018366


## 6) Summary diagnostics

This cell reports aggregate diagnostics to interpret whether clusters are broadly predictive across channels.


In [10]:
eligible = metrics_df[metrics_df["eligible"]].copy()
if len(eligible) == 0:
    print("No channels passed eligibility thresholds.")
else:
    share_significant = (eligible["anova_pvalue"] < 0.05).mean()
    print(f"Eligible channels: {len(eligible)}")
    print(f"Median adj R²: {eligible['adj_r2'].median():.4f}")
    print(f"Mean adj R²:   {eligible['adj_r2'].mean():.4f}")
    print(f"ANOVA significant (p<0.05): {share_significant:.1%}")


Eligible channels: 27
Median adj R²: 0.1077
Mean adj R²:   0.1296
ANOVA significant (p<0.05): 44.4%


## 7) Optional exports (CSV)

This cell saves the complete metric table and ranked view to CSV so results can be shared downstream.


In [11]:
all_metrics_csv = OUTPUT_DIR / "channel_cluster_performance_metrics.csv"
ranked_csv = OUTPUT_DIR / "channel_cluster_performance_ranked_by_adj_r2.csv"

metrics_df.to_csv(all_metrics_csv, index=False)
ranked.to_csv(ranked_csv, index=False)

print(f"Wrote: {all_metrics_csv.resolve()}")
print(f"Wrote: {ranked_csv.resolve()}")


Wrote: /content/drive/MyDrive/Graphiko/analysis/cluster_performance/channel_cluster_performance_metrics.csv
Wrote: /content/drive/MyDrive/Graphiko/analysis/cluster_performance/channel_cluster_performance_ranked_by_adj_r2.csv
